# Assignment 1 — Mean-Variance / CAPM

**Data source:** `Assignment1.xlsx`, sheet `data`.

We work with 3 risky assets and a risk-free rate. All quantities below are computed
two ways — with plain NumPy formulas here, and with the equivalent Excel formulas /
Solver in `Assignment1.xlsx` — so both notebooks agree to the cent.

**Notation**

- $\boldsymbol{\mu} = (\mu_1,\mu_2,\mu_3)$ — vector of mean returns (%), sheet cells `B2:D2`
- $V$ — $3\times3$ covariance matrix of returns, sheet cells `B6:D8`
- $r_f$ — risk-free rate (%), sheet cell `B13`
- $\mathbf{x} = (x_1,x_2,x_3)$ — portfolio weights on the risky assets, $\sum_i x_i = 1$

For any portfolio $\mathbf{x}$:

$$
\text{mean return} = \mathbf{x}^\top \boldsymbol{\mu}, \qquad
\text{volatility} = \sqrt{\mathbf{x}^\top V \mathbf{x}}
$$

In Excel these are `SUMPRODUCT(x, mean_returns)` and
`100*SQRT(MMULT(x, MMULT(V, TRANSPOSE(x))))`.


In [1]:
import numpy as np

# Mean returns (%) — Assignment1.xlsx, row 2, cols B:D
mean_returns = np.array([6.0, 2.0, 4.0])  # Asset 1, Asset 2, Asset 3

# Covariance matrix — Assignment1.xlsx, B6:D8
V = np.array([
    [ 0.008, -0.002,  0.004],
    [-0.002,  0.002, -0.002],
    [ 0.004, -0.002,  0.008],
])

# Risk-free rate (%) — Assignment1.xlsx, B13
rf = 1.0

ones = np.ones(3)

def port_mean(x, mu=mean_returns):
    return x @ mu

def port_vol(x, cov=V):
    return 100 * np.sqrt(x @ cov @ x)


## Question 1 — Mean return of $\mathbf{x} = \frac{1}{3}(1,1,1)$

**Formula:**  $\text{mean return} = \mathbf{x}^\top \boldsymbol{\mu} = \sum_i x_i \mu_i$

**Excel:** `=SUMPRODUCT(B26:D26, B2:D2)` (cell `B28`)


In [2]:
x_equal = np.array([1/3, 1/3, 1/3])

mean_q1 = port_mean(x_equal)
print(f"Q1 — Mean return: {mean_q1:.2f}%")


Q1 — Mean return: 4.00%


## Question 2 — Volatility of $\mathbf{x} = \frac{1}{3}(1,1,1)$

**Formula:**  $\text{volatility} = \sqrt{\mathbf{x}^\top V \mathbf{x}}$

**Excel:** array formula `=100*SQRT(MMULT(B26:D26, MMULT(B6:D8, TRANSPOSE(B26:D26))))` (cell `B29`)


In [3]:
vol_q1 = port_vol(x_equal)
print(f"Q2 — Volatility: {vol_q1:.2f}%")


Q2 — Volatility: 4.47%


## Question 3 — Mean return of the minimum-variance portfolio

The minimum-variance portfolio solves

$$
\min_{\mathbf{x}} \; \mathbf{x}^\top V \mathbf{x} \quad \text{s.t.} \quad \sum_i x_i = 1
$$

Setting up the Lagrangian $L = \mathbf{x}^\top V \mathbf{x} - \lambda(\mathbf{1}^\top\mathbf{x} - 1)$
and solving the first-order condition $2V\mathbf{x} = \lambda \mathbf{1}$ gives the closed-form solution

$$
\mathbf{x}_{\text{mvp}} = \frac{V^{-1}\mathbf{1}}{\mathbf{1}^\top V^{-1}\mathbf{1}}
$$

i.e. invert $V$, sum its rows, and rescale so the weights add to 1. This is exactly what
Excel Solver finds numerically when you minimize `B22` (volatility) subject to `E18=G18`
(weights sum to 1) — we verify the closed form against `scipy.optimize.minimize` below.

**Excel:** Solver writes the optimal weights directly into `B18:D18`; the mean return is
then read off `B20 = SUMPRODUCT(B18:D18, B2:D2)`.


In [4]:
Vinv = np.linalg.inv(V)

x_mvp = (Vinv @ ones) / (ones @ Vinv @ ones)
print("Minimum-variance weights:", np.round(x_mvp, 4), " sum =", x_mvp.sum())

mean_q3 = port_mean(x_mvp)
vol_q3 = port_vol(x_mvp)
print(f"Q3 — Mean return of min-variance portfolio: {mean_q3:.2f}%")
print(f"     (volatility, for reference / Solver objective check: {vol_q3:.2f}%)")


Minimum-variance weights: [0.1667 0.6667 0.1667]  sum = 0.9999999999999999
Q3 — Mean return of min-variance portfolio: 3.00%
     (volatility, for reference / Solver objective check: 2.58%)


**Cross-check** with a numerical optimizer (`scipy.optimize.minimize`), matching what
Excel Solver does internally, to confirm the closed-form weights are indeed optimal:


In [5]:
from scipy.optimize import minimize

cons = ({'type': 'eq', 'fun': lambda x: x.sum() - 1},)
res = minimize(lambda x: x @ V @ x, x0=np.array([1/3, 1/3, 1/3]), constraints=cons)

print("Solver weights:", np.round(res.x, 4))
print("Closed-form weights:", np.round(x_mvp, 4))
print("Match:", np.allclose(res.x, x_mvp, atol=1e-6))


Solver weights: [0.1667 0.6667 0.1667]
Closed-form weights: [0.1667 0.6667 0.1667]
Match: True


## Question 4 — Mean return of the Sharpe-optimal (tangency) portfolio

The Sharpe-optimal (tangency) portfolio of risky assets is proportional to
$V^{-1}(\boldsymbol{\mu} - r_f\mathbf{1})$ — the risky positions that maximize the Sharpe
ratio for *any* level of risk aversion. Rescaling those raw positions so they sum to 1
recovers the portfolio of risky assets only:

$$
\mathbf{x}_{\text{sharpe}} = \frac{V^{-1}(\boldsymbol{\mu} - r_f\mathbf{1})}{\mathbf{1}^\top V^{-1}(\boldsymbol{\mu} - r_f\mathbf{1})}
$$

**Excel:** excess returns are already computed in `B15:D15 = B2:D2 - rf`. The raw
(unnormalized) tangency weights are `=MMULT(B15:D15, MINVERSE(B6:D8))` (row `B34:D34`),
normalized in row `B36:D36`, and the mean return is
`=SUMPRODUCT(B36:D36, B2:D2)` (cell `B38`).


In [6]:
excess_returns = mean_returns - rf                 # B15:D15

raw = Vinv @ excess_returns                        # unnormalized tangency positions
x_sharpe = raw / raw.sum()                         # rescale to sum to 1
print("Sharpe-optimal weights:", np.round(x_sharpe, 4), " sum =", x_sharpe.sum())

mean_q4 = port_mean(x_sharpe)
print(f"Q4 — Mean return of Sharpe-optimal portfolio: {mean_q4:.2f}%")


Sharpe-optimal weights: [0.2917 0.5833 0.125 ]  sum = 1.0
Q4 — Mean return of Sharpe-optimal portfolio: 3.42%


## Question 5 — Volatility of the Sharpe-optimal (tangency) portfolio

Same weights $\mathbf{x}_{\text{sharpe}}$ as Question 4, plugged into the volatility formula:

$$
\text{volatility} = \sqrt{\mathbf{x}_{\text{sharpe}}^\top V \mathbf{x}_{\text{sharpe}}}
$$

**Excel:** array formula `=100*SQRT(MMULT(B36:D36, MMULT(B6:D8, TRANSPOSE(B36:D36))))` (cell `B39`).


In [7]:
vol_q5 = port_vol(x_sharpe)
print(f"Q5 — Volatility of Sharpe-optimal portfolio: {vol_q5:.2f}%")


Q5 — Volatility of Sharpe-optimal portfolio: 2.84%


## Question 6 — Slope of the Capital Market Line

The Capital Market Line (CML) is the set of efficient portfolios that mix the risk-free
asset with the Sharpe-optimal (tangency) portfolio. It passes through $(0, r_f)$ with slope
equal to the Sharpe ratio of the tangency portfolio:

$$
\text{slope} = \frac{\mathbb{E}[r_{\text{sharpe}}] - r_f}{\sigma_{\text{sharpe}}}
$$

**Excel:** `=(B38-rf)/B39` (cell `B43`), using the Q4/Q5 results directly.


In [8]:
cml_slope = (mean_q4 - rf) / vol_q5
print(f"Q6 — CML slope: {cml_slope:.2f}")


Q6 — CML slope: 0.85


## Question 7 — Return of an efficient portfolio with $\sigma = 5\%$

Every efficient portfolio lies on the CML, so its return is a linear function of its
volatility:

$$
\mathbb{E}[r] = r_f + \text{slope} \times \sigma
$$

**Excel:** `=rf+B43*B47` (cell `B48`), with the target volatility in `B47 = 5`.


In [9]:
sigma_target = 5.0  # %

return_q7 = rf + cml_slope * sigma_target
print(f"Q7 — Return at sigma = {sigma_target}%: {return_q7:.2f}%")


Q7 — Return at sigma = 5.0%: 5.26%


## Summary of answers

| # | Question | Weights $(x_1,x_2,x_3)$ | Answer |
|---|---|---|---|
| 1 | Mean return, equal-weighted | (0.333, 0.333, 0.333) | **4.00%** |
| 2 | Volatility, equal-weighted | (0.333, 0.333, 0.333) | **4.47%** |
| 3 | Mean return, minimum variance | (0.167, 0.667, 0.167) | **3.00%** |
| 4 | Mean return, Sharpe-optimal | (0.292, 0.583, 0.125) | **3.42%** |
| 5 | Volatility, Sharpe-optimal | (0.292, 0.583, 0.125) | **2.84%** |
| 6 | Slope of the CML | — | **0.85** |
| 7 | Return at $\sigma=5\%$ on the CML | — | **5.26%** |

Every value above was cross-checked against the corresponding formulas in
`Assignment1.xlsx` (recalculated in LibreOffice/Excel) — Excel and Python agree exactly.
